In [109]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [110]:
def clean_column_names(df):
    """Очищает имена колонок от символов, которые не принимает XGBoost"""
    new_cols = []
    for col in df.columns:
        c = col.replace("[", "_").replace("]", "_").replace("<", "_lt_")
        c = c.replace(">", "_gt_").replace(" ", "_").replace(":", "_")
        c = c.replace("(", "_").replace(")", "_").replace(",", "_")
        c = c.replace("{", "_").replace("}", "_").replace("|", "_")
        c = "_".join(filter(None, c.split("_"))).strip("_")
        if c and c[0].isdigit():
            c = "f_" + c
        new_cols.append(c)
    df_clean = df.copy()
    df_clean.columns = new_cols
    return df_clean

def preprocess_data(df, target_col="Смерть", drop_cols=None, group_col="Name"):
    """Полный цикл препроцессинга: кодирование таргета, очистка колонок, LabelEncoding, расчет весов"""
    df = df.copy()
    df[target_col] = df[target_col].map({"Да": 1, "Нет": 0})
    df.dropna(subset=[target_col], inplace=True)
    
    X = df.drop(columns=[target_col])
    y = df[target_col]
    X_clean = clean_column_names(X)
    
    object_cols = X_clean.select_dtypes(include=["object"]).columns
    for col in object_cols:
        le = LabelEncoder()
        combined = pd.concat([X_clean[col], X_clean[col]], axis=0).astype(str)  # сохранено как в оригинале
        le.fit(combined)
        X_clean[col] = le.transform(X_clean[col].astype(str))
        
    if drop_cols:
        X_clean.drop(columns=drop_cols, inplace=True, errors="ignore")
        
    scale_pos_weight = len(y[y == 0]) / max(1, len(y[y == 1]))
    class_weight_dict = {0: 1, 1: scale_pos_weight}
    
    groups = X_clean[group_col].astype(str).str.strip().str.upper() if group_col in X_clean.columns else None
    X_cv = X_clean.drop(columns=[group_col] + (drop_cols or []), errors="ignore").copy()
    
    return X_cv, y, groups, scale_pos_weight, class_weight_dict

def fmt_ci(scores):
    m = np.mean(scores)
    ci_l, ci_h = np.percentile(scores, [2.5, 97.5])
    return f"{m:.3f} [{ci_l:.3f}–{ci_h:.3f}]"

def run_cv_and_print(models, X, y, cv, groups=None, title="CV Results"):
    scorers = {
        "pr_auc": "average_precision", "roc_auc": "roc_auc",
        "f1": make_scorer(f1_score, pos_label=1), "bal_acc": make_scorer(balanced_accuracy_score)
    }
    results = {}
    cv_kwargs = {"groups": groups} if groups is not None else {}
    
    for name, model in models.items():
        print(f"\n🔄 {title} для {name}...")
        results[name] = cross_validate(model, X, y, cv=cv, scoring=scorers, n_jobs=-1, return_train_score=False, **cv_kwargs)
        
    print("\n" + "=" * 80)
    print(f"📈 {title}")
    print("=" * 80)
    print(f"{'Model':<12} | {'PR-AUC':<18} | {'ROC-AUC':<18} | {'F1':<18} | {'Bal Acc':<18}")
    print("-" * 80)
    for mdl, res in results.items():
        print(f"{mdl:<12} | {fmt_ci(res['test_pr_auc']):<18} | {fmt_ci(res['test_roc_auc']):<18} | "
              f"{fmt_ci(res['test_f1']):<18} | {fmt_ci(res['test_bal_acc']):<18}")
    return results

In [111]:
data = pd.read_csv("./data_raw/clean_table.csv", index_col=0)
X_cv, y, groups, spw, cwd = preprocess_data(data, drop_cols=["Код_пациента"])

/tmp/ipykernel_130640/3844404189.py:1: DtypeWarning: Columns (34,35,41) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("./data_raw/clean_table.csv", index_col=0)


In [112]:
cv_rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
models_default = {
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=spw, random_state=42, eval_metric="logloss", tree_method="auto", enable_categorical=True),
    "LightGBM": LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, class_weight=cwd, random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=100, depth=5, learning_rate=0.1, class_weights=[1, spw], random_state=42, verbose=False, eval_metric="Logloss")
}
run_cv_and_print(models_default, X_cv, y, cv=cv_rskf, title="Честные метрики (Repeated Stratified 5-Fold CV, 10 повторов = 50 оценок)")


🔄 Честные метрики (Repeated Stratified 5-Fold CV, 10 повторов = 50 оценок) для XGBoost...

🔄 Честные метрики (Repeated Stratified 5-Fold CV, 10 повторов = 50 оценок) для LightGBM...

🔄 Честные метрики (Repeated Stratified 5-Fold CV, 10 повторов = 50 оценок) для CatBoost...


AttributeError: The following error was raised: 'CatBoostClassifier' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.